# 01 获取标准条件阳性数据

按照 `Table of Content` 合并标准 lysate、标准 in situ 和标准 1 mM 工作表。带重复编号、额外处理或特殊修饰条件后缀的工作表全部排除并另存清单。

In [8]:
# Cell 1：导入库并设置文件路径
import os
import re
import openpyxl
import pandas as pd

input_file = os.path.join("raw_data", "41557_2025_1902_MOESM6_ESM.xlsx")
output_dir = os.path.join("outputs", "positive_data")
output_file = os.path.join(output_dir, "merged_data.csv")

In [9]:
# Cell 2：读取工作簿和 Table of Content
# 浓度字符已经修正，所以这里直接读取单元格文本即可。
workbook = openpyxl.load_workbook(input_file, read_only=True, data_only=True)
toc = pd.read_excel(input_file, sheet_name="Table of Content", usecols=["Worksheet", "Information"])

# 目录末尾还有字段说明，先只保留确实存在的工作表。
data_sheet_names = set(workbook.sheetnames) - {"Table of Content"}
toc = toc[toc["Worksheet"].isin(data_sheet_names)].copy()

# 只保留标准 lysate、标准 in situ 和标准 1 mM；带其他后缀的特殊条件全部排除。
canonical_pattern = re.compile(r"^(?:SA|HS) .+-alkyne(?: in situ| 1 mM)?$")
keep_mask = toc["Worksheet"].astype(str).str.fullmatch(canonical_pattern)
excluded_toc = toc[~keep_mask].copy()
toc = toc[keep_mask].copy()

print(f"标准工作表：{len(toc)} 个；排除特殊条件工作表：{len(excluded_toc)} 个")
display(excluded_toc[["Worksheet", "Information"]])
toc

标准工作表：89 个；排除特殊条件工作表：28 个


,Worksheet,Information
0,SA IA-alkyne 1 mM 1-1,Amino acid selectivity for IA-alkyne at 1 mM p...
1,SA IA-alkyne 1 mM 4-1,Amino acid selectivity for IA-alkyne at 1 mM p...
3,SA IA-alkyne no enrichment,Amino acid selectivity for IA-alkyne at 100 µM...
14,SA EBX1-alkyne exp,Amino acid selectivity for EBX1-alkyne at 100 ...
15,SA EBX1-alkyne VI,Amino acid selectivity for EBX1-alkyne at 100 ...
16,SA EBX1-alkyne VBX,Amino acid selectivity for EBX1-alkyne at 100 ...
61,SA MeTet-alkyne 5 min irr.,Amino acid selectivity for MeTet-alkyne at 100...
62,SA MeTet-alkyne 30 min irr exp,Amino acid selectivity for MeTet-alkyne at 100...
63,SA MeTet-alkyne 30 min irr Aryl,Amino acid selectivity for MeTet-alkyne at 100...
68,SA Az-alkyne comb,Amino acid selectivity for Az-alkyne at 100 µM...


,Worksheet,Information
2,SA IA-alkyne,Amino acid selectivity for IA-alkyne at 100 µM...
4,SA IA-alkyne in situ,Amino acid selectivity for IA-alkyne at 100 µM...
5,SA IA-alkyne 1 mM,Amino acid selectivity for IA-alkyne at 1 mM p...
6,HS IA-alkyne,Amino acid selectivity for IA-alkyne at 100 µM...
7,SA CA-alkyne,Amino acid selectivity for CA-alkyne at 100 µM...
...,...,...
108,HS MMP-alkyne,Amino acid selectivity for MMP-alkyne at 100 µ...
113,SA PhGO-alkyne in situ,Amino acid selectivity for PhGO-alkyne at 100 ...
114,HS PhGO-alkyne 1 mM,Amino acid selectivity for PhGO-alkyne at 1 mM...
115,SA FP-alkyne,Amino acid selectivity for FP-alkyne at 100 µM...


In [10]:
# Cell 3：定义 Information 和 Identifier 的解析函数
information_pattern = re.compile(
    r"^Amino acid selectivity for (?P<probe>.+?) "
    r"at (?P<concentration>[0-9.]+\s*(?:µM|μM|mM|uM)) "
    r"probe concentration in the proteome of "
    r"(?P<cell>S\. aureus SH1000|the human cancer cell line MDA-MB-231)"
    r"(?P<condition>.*)$",
    flags=re.IGNORECASE,
)

identifier_pattern = re.compile(r"^(.+)_([A-Z]|N-terminal|C-terminal)(\d+)$")

def parse_information(text):
    match = information_pattern.fullmatch(str(text).strip())
    if not match:
        raise ValueError(f"无法解析 Information: {text}")

    probe = match.group("probe")
    concentration = match.group("concentration").replace("μ", "µ")
    cell_line = "MDA-MB-231" if "MDA-MB-231" in match.group("cell") else "S. aureus SH1000"
    condition = re.sub(r"\s+", " ", match.group("condition").strip())
    modification_condition = condition if condition else "standard"

    # 将 Information 中的附加条件进一步拆成独立字段。
    treatment = "in situ" if "probe treatment in situ" in condition else "lysate"

    if "5 min irradiation" in condition:
        irradiation = "5 min irradiation"
    elif "30 min irradiation" in condition:
        irradiation = "30 min irradiation"
    elif "irradiation open to air" in condition:
        irradiation = "irradiation open to air"
    elif "irradiation under Argon" in condition:
        irradiation = "irradiation under Argon"
    else:
        irradiation = "not specified"

    if "expected modification by alkynylation" in condition:
        modification_mass = "expected modification by alkynylation"
    elif "vinyl iodide" in condition:
        modification_mass = "vinyl iodide"
    elif "vinylbenziodoxolone" in condition:
        modification_mass = "vinylbenziodoxolone"
    elif "diacylated hydrazine" in condition:
        modification_mass = "expected modification as diacylated hydrazine"
    elif "modification by arylation" in condition:
        modification_mass = "arylation"
    elif "expected modifcation and the expected modification +1 Da" in condition:
        modification_mass = "expected modification and expected modification +1 Da"
    elif "expected modifcation + 1 Da" in condition:
        modification_mass = "expected modification +1 Da"
    elif "reactivity as isocyanate" in condition:
        modification_mass = "isocyanate product"
    elif "oxidation product" in condition:
        modification_mass = "oxidation product"
    elif "expected modifcation" in condition or "expected modification" in condition:
        modification_mass = "expected modification"
    else:
        modification_mass = "not specified"

    if "without clicking and without enrichment" in condition:
        sample_preparation = "without clicking and without enrichment"
    elif "200 mM Tris-HCl pH = 7.4" in condition:
        sample_preparation = "200 mM Tris-HCl pH = 7.4 added to sample"
    elif "Zeba Spin purification" in condition:
        sample_preparation = "Zeba Spin purification before CuAAC to isoDTB tags"
    else:
        sample_preparation = "standard"

    ratio_match = re.search(r"mixed at a ratio of (\d+:\d+)", condition)
    mixing_ratio = ratio_match.group(1) if ratio_match else "not specified"
    dataset_note = "pre-published data set" if "pre-published data set" in condition else "not specified"

    return {
        "Probe": probe,
        "Cell_line": cell_line,
        "Concentration": concentration,
        "Treatment": treatment,
        "Irradiation_condition": irradiation,
        "Modification_mass_condition": modification_mass,
        "Sample_preparation": sample_preparation,
        "Mixing_ratio": mixing_ratio,
        "Dataset_note": dataset_note,
        "Modification_condition": modification_condition,
    }

In [11]:
# Cell 4：按目录顺序读取并合并所有工作表
all_tables = []
sheet_tables = pd.read_excel(input_file, sheet_name=toc["Worksheet"].tolist())

for toc_order, row in enumerate(toc.itertuples(index=False), start=1):
    worksheet = row.Worksheet
    information = row.Information
    conditions = parse_information(information)

    table = sheet_tables[worksheet].copy()
    table = table.dropna(how="all")

    # 从 Identifier 解析 UniProt、位点类型和位置。
    parsed = table["Identifier"].astype(str).str.extract(identifier_pattern)
    parsed.columns = ["UNIPROT_ID", "Site_type", "Position"]
    parsed["Position"] = parsed["Position"].astype("Int64")

    # 在表格前面加入目录和实验信息。
    table.insert(0, "toc_order", toc_order)
    table.insert(1, "Worksheet", worksheet)
    table.insert(2, "Information", information)
    # 把 Information 中解析出的所有实验条件加入数据。
    insert_at = 3
    for column, value in conditions.items():
        table.insert(insert_at, column, value)
        insert_at += 1
    table["UNIPROT_ID"] = parsed["UNIPROT_ID"]
    table["Position"] = parsed["Position"]
    table["Site_type"] = parsed["Site_type"]

    all_tables.append(table)

merged = pd.concat(all_tables, ignore_index=True)
print(f"合并完成：{len(merged):,} 条记录")
merged.head()

合并完成：85,512 条记录


,toc_order,Worksheet,Information,Probe,Cell_line,Concentration,Treatment,Irradiation_condition,Modification_mass_condition,Sample_preparation,...,Modification_condition,Identifier,Modified residue,UniProt code,Gene name,Name,Essential?,UNIPROT_ID,Position,Site_type
0,1,SA IA-alkyne,Amino acid selectivity for IA-alkyne at 100 µM...,IA-alkyne,S. aureus SH1000,100 µM,lysate,not specified,not specified,standard,...,standard,A0A0H3KE27_C18,C,A0A0H3KE27,rsbU,sigma factor sigB regulation protein,no,A0A0H3KE27,18,C
1,1,SA IA-alkyne,Amino acid selectivity for IA-alkyne at 100 µM...,IA-alkyne,S. aureus SH1000,100 µM,lysate,not specified,not specified,standard,...,standard,A0A0H3KE27_C28,C,A0A0H3KE27,rsbU,sigma factor sigB regulation protein,no,A0A0H3KE27,28,C
2,1,SA IA-alkyne,Amino acid selectivity for IA-alkyne at 100 µM...,IA-alkyne,S. aureus SH1000,100 µM,lysate,not specified,not specified,standard,...,standard,A0A0H3KE27_M171,M,A0A0H3KE27,rsbU,sigma factor sigB regulation protein,no,A0A0H3KE27,171,M
3,1,SA IA-alkyne,Amino acid selectivity for IA-alkyne at 100 µM...,IA-alkyne,S. aureus SH1000,100 µM,lysate,not specified,not specified,standard,...,standard,O05154_C10,C,O05154,tagX,Putative glycosyltransferase TagX,no,O05154,10,C
4,1,SA IA-alkyne,Amino acid selectivity for IA-alkyne at 100 µM...,IA-alkyne,S. aureus SH1000,100 µM,lysate,not specified,not specified,standard,...,standard,O05154_C69,C,O05154,tagX,Putative glycosyltransferase TagX,no,O05154,69,C


In [12]:
# Cell 5：检查合并结果
assert len(toc) == 89, f"标准工作表不是89个，而是 {len(toc)} 个"
assert len(excluded_toc) == 28, f"排除工作表不是28个，而是 {len(excluded_toc)} 个"
assert len(merged) == 85512, f"合并记录数不是85512，而是 {len(merged)}"
assert merged["Cell_line"].notna().all()
assert merged["Concentration"].notna().all()
assert merged["Probe"].notna().all()
condition_columns = ["Probe", "Cell_line", "Concentration", "Treatment", "Irradiation_condition", "Modification_mass_condition", "Sample_preparation", "Mixing_ratio", "Dataset_note", "Modification_condition"]
assert merged[condition_columns].notna().all().all()
assert merged["UNIPROT_ID"].notna().all()
assert merged["Position"].notna().all()
assert (merged["Modified residue"] == merged["Site_type"]).all()

merged.groupby(["Cell_line", "Concentration"]).size().rename("记录数")

Cell_line         Concentration
MDA-MB-231        1 mM               462
                  100 µM           16597
S. aureus SH1000  1 mM             18103
                  100 µM           50350
Name: 记录数, dtype: int64

In [13]:
# Cell 6：添加三组标签，并保存唯一的总表
if "Dataset_group" in merged.columns:
    merged = merged.drop(columns="Dataset_group")

dataset_group = pd.Series(index=merged.index, dtype="string")
dataset_group.loc[merged["Cell_line"] == "MDA-MB-231"] = "Human_Lysate"
dataset_group.loc[(merged["Cell_line"] == "S. aureus SH1000") & (merged["Treatment"] == "lysate")] = "SA59_Lysate"
dataset_group.loc[(merged["Cell_line"] == "S. aureus SH1000") & (merged["Treatment"] == "in situ")] = "SA59_Insitu"

assert dataset_group.notna().all()
merged.insert(3, "Dataset_group", dataset_group)

os.makedirs(output_dir, exist_ok=True)
merged.to_csv(output_file, index=False, encoding="utf-8-sig")
excluded_toc.to_csv(os.path.join(output_dir, "excluded_worksheets.csv"), index=False, encoding="utf-8-sig")
print(f"已保存：{os.path.abspath(output_file)}")

已保存：e:\ChenClaw\ProbPred\outputs\positive_data\merged_data.csv


In [14]:
# Cell 7：按三组标签和浓度直接输出5个子表

concentration_labels = {
    "100 µM": "100uM",
    "1 mM": "1mM",
}

subsets = {}

for (group_name, concentration), data in merged.groupby(["Dataset_group", "Concentration"], sort=False):
    concentration_label = concentration_labels[concentration]
    subset_name = f"{group_name}_{concentration_label}"
    # 保留 Concentration，方便从子表中的每一行追溯实验浓度。
    subset = data.drop(columns=["Dataset_group", "Cell_line", "Treatment"]).copy()
    subsets[subset_name] = subset

    save_path = os.path.join(output_dir, f"{subset_name}.csv")
    subset.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"{subset_name}: {len(subset):,} 条 -> {save_path}")

# 5个非空浓度数据集必须完整覆盖原始三组数据。
assert len(subsets) == 5
assert sum(len(data) for data in subsets.values()) == len(merged)
assert len(subsets["Human_Lysate_100uM"]) == 16597
assert len(subsets["Human_Lysate_1mM"]) == 462
assert len(subsets["SA59_Lysate_100uM"]) == 43559
assert len(subsets["SA59_Lysate_1mM"]) == 18103
assert len(subsets["SA59_Insitu_100uM"]) == 6791
assert "SA59_Insitu_1mM" not in subsets

SA59_Lysate_100uM: 43,559 条 -> outputs\positive_data\SA59_Lysate_100uM.csv
SA59_Insitu_100uM: 6,791 条 -> outputs\positive_data\SA59_Insitu_100uM.csv
SA59_Lysate_1mM: 18,103 条 -> outputs\positive_data\SA59_Lysate_1mM.csv
Human_Lysate_100uM: 16,597 条 -> outputs\positive_data\Human_Lysate_100uM.csv
Human_Lysate_1mM: 462 条 -> outputs\positive_data\Human_Lysate_1mM.csv
